# Tutorial 02: Loading and Inspecting Your Images

**About this tutorial**
- **Time:** ~20 minutes
- **Prerequisites:** PhenoMe environment installed ([Getting started](../../docs/src/content/docs/getting-started.md))
- **Key ideas:** `find_files`, `metadata_fn`, `inspect_data`, folder layouts, CSV lookup, multi-channel images.

**Before you start:** Run this notebook from the repo (e.g. open it under `Notebooks/tutorials/`). Replace every `path/to/...` and adjust the path template `.../(condition)/(filename).*` to match your folder layout.

**Why prepare and inspect?** Microscopes sometimes save images in different sizes or formats, and occasionally masks don't perfectly align with their images. This step acts as a safety check to spot irregularities before running the heavy AI analysis, saving you time and preventing confusing errors later on.

**Note:** You can use the E. coli toy dataset from [Tutorial 03](03_core_phenotyping_workflow.ipynb) to test these steps out-of-the-box.

See [Best Practices: Data Inspection](../../docs/src/content/docs/guides/best-practices.md#data-inspecting) and [Getting started](../../docs/src/content/docs/getting-started.md).

**Author:** Aitor González-Marfil ([@AAitorG](https://github.com/AAitorG))

---

## 1. Setup

In [ ]:
from phenome import PhenoMe, get_metadata_from_path, make_dataframe_metadata_fn
import pandas as pd

pheno = PhenoMe()

## 2. Loading Your Data and Metadata

Before inspecting or processing, you need to tell PhenoMe where your files are and what they represent (their metadata). The `find_files()` method is the entry point for this.

### 2.1 Method A: Path Templates (Folders as labels)
Use this when your folder structure already contains the experimental labels (e.g., `data/Control/img01.tif`).

In [ ]:
image_dir = "path/to/images"
metadata_fn = get_metadata_from_path(".../(condition)/(filename).*")

file_df = pheno.find_files(
    image_dir,
    metadata_fn=metadata_fn,
    extensions=[".tif", ".tiff", ".png"],
)

print(f"Found {len(file_df)} images using path template")
print(file_df.head())

### 2.2 Method B: CSV Lookup (External spreadsheets)
Use this when your labels live in an external CSV or Excel file. PhenoMe will match files on disk to rows in your CSV, usually by filename.

In [ ]:
# Load your spreadsheet
# df = pd.read_csv("metadata.csv")
df = pd.DataFrame({
    'filename': ['image_001', 'image_002'],
    'treatment': ['Control', 'DrugA'],
    'dose': [0, 10]
})

# Create the lookup function
metadata_fn = make_dataframe_metadata_fn(df, filename_column='filename')

file_df = pheno.find_files("path/to/images", metadata_fn=metadata_fn)
print(f"Found {len(file_df)} images using CSV lookup")

### 2.3 Method C: Multi-channel (Each channel in a different file)
Some microscopes save colors separately (e.g., `wellA1_ch0.tif`, `wellA1_ch1.tif`). PhenoMe can group these into a single multi-channel sample.

In [ ]:
df_channels = pd.DataFrame({
    'ch0': ['sample1_c0.tif', 'sample2_c0.tif'],
    'ch1': ['sample1_c1.tif', 'sample2_c1.tif'],
    'condition': ['Control', 'Treated']
})

# Order matters: [ch0, ch1, ...]
metadata_fn = make_dataframe_metadata_fn(df_channels, filename_column=['ch0', 'ch1'])

file_df = pheno.find_files("path/to/images", metadata_fn=metadata_fn)
print(f"Found {len(file_df)} multi-channel samples")

## 3. Run Inspection and Interpret Results

Now that your files are loaded, run `inspect_data()` to check for errors, dimension mismatches, or mask misalignments.

Key things it checks:

- **Dimensions & Channels:** Matches their height, width, and number of colors.
- **Errors:** Finds corrupted files or broken paths before they crash the AI.
- **shape_match:** Checks if your image and its mask perfectly overlap. If `False`, it means they are different sizes and need fixing before analysis.

**What to look for:** Most of your images should have the same dimensions. If not, you might need to crop or pad them during processing to make them consistent.

| Finding | Typical Action (when running process_images) |
|---------|---------------------------------------------|
| Common image size | The default `resize_size=224` usually works well. |
| Very small images | Consider setting `pad_size` before resizing to avoid stretching them out. |
| Multi-channel fluorescence | Use `channel_mode='split'` so the AI handles colors correctly. |
| General photography (RGB) | Use `channel_mode='combined'` as you would with standard photos. |

In [ ]:
inspect_df = pheno.inspect_data()
print(inspect_df.head(10))

## 4. Proceed to Processing

After inspecting, run `process_images()` with appropriate parameters. See [Common Workflows](../../docs/src/content/docs/workflows.md) for full examples.

In [ ]:
# Example: process with parameters informed by inspect
# from phenome import load_dinov2_model
# wrapper = load_dinov2_model(device=device)
# pheno.process_images(wrapper, resize_size=224, channel_mode="split")

## Next Steps

- [03_core_phenotyping_workflow.ipynb](03_core_phenotyping_workflow.ipynb): Core API tutorial
- [04_advanced_metadata_handling.ipynb](04_advanced_metadata_handling.ipynb): Handling messy data
- [05_exploratory_analysis_and_explainability.ipynb](05_exploratory_analysis_and_explainability.ipynb): Advanced analysis after successful QC and embedding
- [Best Practices](../../docs/src/content/docs/guides/best-practices.md): Reproducibility and error handling